# Beni wake-word training (Colab) — V4 lean-T4

End-to-end notebook that trains a microWakeWord model for the phrase
**"Hey Beni"** and exports a streaming int8 TFLite the ESP32-S3
firmware can load directly.

This notebook is a **thin adaptation** of the upstream reference at
`kahrendt/microWakeWord/notebooks/basic_training_notebook.ipynb`.
If you need to tune something and this notebook doesn't cover it,
diff against upstream — we intentionally stay close to it so
cross-referencing the docs works.

**V4 changes vs. V3** (2026-04-29) — all aimed at fitting comfortably
on a free-tier T4 (15 GB VRAM, ~12 GB system RAM, ~12 h max session):

* Cell 2 now **hard-fails if no GPU is visible**, instead of a soft
  warning. Saves ~25 min of pointless data-prep on CPU before the
  trainer crashes.
* Cell 2 sets `TF_FORCE_GPU_ALLOW_GROWTH=true` so VRAM is allocated
  lazily — earlier full-grab caused parallel imports to OOM.
* Positives reduced from 10 000 → **4 000**. piper-sample-generator's
  multi-speaker bank is fully covered at 4 k; the diversity ceiling
  is the speaker count, not the sample count.
* Negative-dataset zips: dropped `dinner_party_eval.zip` (eval-only
  fixture, not consumed by the main trainer; saves ~1.5 GB disk +
  the unzip pass).
* Trainer schedule **18 k steps total** (was 60 k), batch 96 (was
  128), pointwise filters 48–48–48–64 (was 64×4). Same MixedNet
  topology, smaller width — fits T4 with headroom and trains in
  ~50 min instead of ~2 h.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (or A100 if available).
2. Click Connect (top-right). Wait for the green check + "T4" label
   before running anything.
3. Run all cells top-to-bottom. After the install cell you'll need
   to **Restart Session** — Colab will prompt.
4. The final cell downloads `stream_state_internal_quant.tflite`
   and a ready-to-include `hey_beni_model.h`. Drop both into
   `firmware/main/wake_word/model/`.

Total runtime: roughly **45–60 min on a T4** end-to-end (~15 min
data prep + ~5 min negative-dataset download + ~30 min training +
exports). On a free-tier session the 12 h limit is no longer in
play — even two consecutive runs fit comfortably.

## 1. Install microWakeWord

The package is not on PyPI — it's distributed from the author's
GitHub. We also pull a patched `audio-metadata` (upstream broke on
recent `attrs` versions, the fork un-pins it) to avoid a cryptic
import error during data loading.

**After this cell finishes, Colab will ask to restart the session.
Do it.** TensorFlow is reloaded from microwakeword's pinned version
and the current kernel has stale C-extensions.

In [ ]:
import os, platform

if platform.system() == 'Darwin':
    # Not our target env (Colab = Linux) but handy if a dev runs this
    # locally on a Mac for smoke testing.
    !pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

# `attrs` fork keeps audio-metadata working across TF's pinned attrs.
!pip install -q 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

# Idempotent clone + editable install. After a runtime rebuild the
# filesystem is wiped, so re-running this cell must succeed without
# manual cleanup — hence the existence check around git clone.
if not os.path.exists('microWakeWord'):
    !git clone https://github.com/kahrendt/microWakeWord
!pip install -q -e ./microWakeWord

# --- Patch microwakeword for Keras 3 / TF 2.16+ ---
# train.py calls .numpy() on the result dict from model.evaluate(return_dict=True).
# In TF 2.16+ Keras 3, evaluate already returns numpy arrays/scalars, so
# calling .numpy() on them raises AttributeError. The subprocess that runs
# training reads the file fresh from disk, so patching the source file is
# sufficient (no reload dance needed in this kernel).
_train_py = 'microWakeWord/microwakeword/train.py'
_src = open(_train_py).read()
for _old, _new in [
    ('result["fp"].numpy()', 'np.asarray(result["fp"])'),
    ('ambient_predictions["tp"].numpy()', 'np.asarray(ambient_predictions["tp"])'),
    ('ambient_predictions["fp"].numpy() - test_set_fp', 'np.asarray(ambient_predictions["fp"]) - test_set_fp'),
    ('ambient_predictions["fn"].numpy()', 'np.asarray(ambient_predictions["fn"])'),
]:
    _src = _src.replace(_old, _new)
open(_train_py, 'w').write(_src)
print('patched train.py for Keras 3 evaluate() return shape')

# Pin datasets to 2.x: microwakeword's audio pipeline (clips.py) and
# our data-download cells both access audio rows via dict subscript
# (`row["audio"]["array"]`). `datasets` 3.x switched to a torchcodec
# `AudioDecoder` object which is not subscriptable, so the old
# pattern crashes with `TypeError: 'AudioDecoder' object is not
# subscriptable`. Downgrading is cleaner than rewriting every
# subscript site, and torchcodec gets uninstalled so datasets falls
# back to soundfile.
!pip install -q 'datasets<3.0.0'
!pip uninstall -y -q torchcodec || true

# microwakeword's setup.py pins audiomentations with no minimum version,
# and Colab ships an older copy that lacks `AddColorNoise` (added in
# 0.34). Augmentation() in cell 10 refers to it unconditionally, so
# force-upgrade here.
!pip install -q -U 'audiomentations>=0.35'

# --- GPU gate + TF runtime tuning ---
#
# The training subprocess in cell 18 (`!python -m microwakeword.
# model_train_eval`) inherits THIS kernel's environment, so any
# TF_* env vars set here propagate into the trainer. We use that
# rather than patching microwakeword for runtime tweaks.
#
# Hard-fail on no-GPU: 4 k positives × full augmentation × 18 k
# training steps takes ~50 min on T4 and a multi-day affair on
# Colab CPU. Better to crash here than 25 min into data prep —
# the previous "soft warn" path led to several wasted sessions.
import subprocess as _sp
_r = _sp.run(['nvidia-smi'], capture_output=True, text=True)
if _r.returncode != 0:
    raise RuntimeError(
        'No NVIDIA GPU visible.\n'
        '  1. Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save.\n'
        '  2. Click Connect (top-right). Wait for the green check + "T4" label.\n'
        '  3. Re-run THIS cell.\n'
        'If T4 shows "unavailable", you hit the free-tier daily quota — '
        'wait 12-24 h, switch Google account, or upgrade to Colab Pro.'
    )
# Print the line with the GPU name + driver/CUDA versions, not the
# full table (clutters Colab output).
_lines = _r.stdout.split('\n')
print(_lines[7] if len(_lines) > 7 else _r.stdout[:200])

# Lazy VRAM allocation. Without this TF grabs the full 15 GB on
# first op, which (a) blocks any parallel notebook from importing
# TF and (b) makes Colab's memory monitor scream "leak". Costs
# nothing, helps everything.
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

# Mute TF's verbose info log (XLA spam, NUMA hints, AVX hints on
# cloud CPUs). Real errors and warnings still print.
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

print('GPU OK. TF_FORCE_GPU_ALLOW_GROWTH=true; TF_CPP_MIN_LOG_LEVEL=2')

## 2. Generate 1 sample of "Hey Beni" to verify Piper settings

piper-sample-generator takes an English phrase and renders many
voiced variants by sampling from the LibriTTS neural voice. "beni"
reads phonetically correctly in English ("BEN-ee") so we don't need
a Russian voice for v1 — the wake detector is acoustic, not
linguistic. If later we want a Russian voice, swap the model file
below for a Piper ru voice.

Listen to the output of this cell to sanity-check the pronunciation
**before** generating thousands of samples.

In [ ]:
import os
from IPython.display import Audio

# --- NOTEBOOK REVISION V4 lean-T4 -------------------------------
# If the next printed line below doesn't show this hash or newer,
# Colab is running a stale cached copy. File -> Revert, or close
# and reopen the tab from the GitHub URL.
print('notebook revision: V4 lean-T4 (4k positives, 18k steps, batch 96, 48-48-48-64 filters)')
# -----------------------------------------------------------------

TARGET_WORD = 'hey beni'  # Phonetic; matches how callers will say it

# piper-sample-generator is on PyPI, but v3.2.0's wheel is broken:
# `pyproject.toml` restricts packaging to `piper_sample_generator*`,
# which excludes the sibling `piper_train` package — yet
# `piper_sample_generator.__main__` imports from it (`from
# piper_train.vits import commons`). Separately, the CLI expects a
# `<model>.pt.json` config next to the .pt, which is only shipped
# inside the source repo's `models/` directory, not the release .pt.
#
# So: install the PyPI wheel for its runtime deps, clone the repo
# once for both the missing `piper_train` subpackage and the .json
# configs, and download the .pt release artifact into that same
# `models/` directory so the CLI finds its paired config.
!pip install -q piper-sample-generator

# Kill any leftover state from earlier notebook revisions that put
# the model under ./piper_models/ (without its .json sibling).
!rm -rf piper_models

PSG_SRC = '/tmp/psg_src'
if not os.path.exists(PSG_SRC):
    !git clone --depth 1 https://github.com/rhasspy/piper-sample-generator {PSG_SRC}

# cwd goes first on sys.path for `python3 -m`, so a sibling
# `piper_train/` here resolves the otherwise-failing import.
if not os.path.exists('piper_train'):
    !cp -r {PSG_SRC}/piper_train .

# Pair the .pt release artifact with its .json config (already in the
# clone). The CLI reads `{model_path}.json` unconditionally.
MODEL_PATH = f'{PSG_SRC}/models/en_US-libritts_r-medium.pt'
if not os.path.exists(MODEL_PATH):
    !wget -q -O {MODEL_PATH} \
        'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

# Sanity: confirm both files are colocated before invoking the CLI.
assert os.path.exists(MODEL_PATH), f'missing .pt at {MODEL_PATH}'
assert os.path.exists(MODEL_PATH + '.json'), f'missing .json at {MODEL_PATH}.json'
print('model OK:', MODEL_PATH)

!python3 -m piper_sample_generator "{TARGET_WORD}" \
    --model {MODEL_PATH} \
    --max-samples 1 --batch-size 1 --output-dir generated_samples

Audio('generated_samples/0.wav', autoplay=True)

## 3. Generate 4 000 wake-word samples (multi-speaker, default prosody)


In [ ]:
# Generate 4 000 positive samples for "hey beni".
#
# The libritts_r-medium fine-tune that piper-sample-generator ships is a
# *multi-speaker* model — it iterates over its speaker bank automatically
# across batches, so 4 k samples already pulls from hundreds of distinct
# voices. V3 used 10 k; experiments showed no recall gain beyond ~4 k
# because we hit the speaker-diversity ceiling — extra clips are
# near-duplicates of voices we've already seen. Cutting to 4 k saves
# ~12 min in this cell and ~7 min in the spectrogram pass below, all
# without hurting the trained model's FRR/FAR table.
#
# We deliberately do NOT pass `--noise-scales` / `--length-scales` /
# `--noise-scale-ws` here because their behaviour in the version of
# piper-sample-generator pinned in cell 4 is uncertain (V2 of this
# notebook tried that and got a peak-score regression of 0.31 → 0.035
# on the trained model — possibly the flags were silently ignored,
# possibly they over-stretched the audio past clip_duration_ms).
!python3 -m piper_sample_generator "{TARGET_WORD}" \
    --model {MODEL_PATH} \
    --max-samples 4000 --batch-size 100 \
    --output-dir generated_samples

import glob
print('positive samples:', len(glob.glob('generated_samples/*.wav')))


## 4. Download augmentation data (RIRs + background noise)

Two sources, both on HuggingFace:

* **MIT impulse responses** — convolved with positives to simulate
  room reverb. Makes the model robust to hearing "Hey Beni" from
  across a kitchen vs. directly into the mic.
* **ESC-50** — 2000 environmental-sound clips (motors, water,
  animals, keyboard clacks, etc). Mixed into positives at varied
  SNR so the model learns the wake word is "Hey Beni + whatever
  else".

Upstream notebook also used AudioSet + FMA, but both have rotted on
HF (AudioSet's shard URL now serves HTML 404s, `mchl914/free_music_archive_xs`
was deleted). ESC-50 is smaller and more reliable — we can add a
music dataset back in later if false-accepts on TV/radio are bad.

Speech-like distractors are **not** needed here — they come from
the pre-generated `speech.zip` negative spectrograms in cell 7,
which is the proper training signal for "this is speech but not the
wake word"."

In [ ]:
import datasets, scipy, os, numpy as np
from pathlib import Path
from tqdm import tqdm

# --- MIT RIRs (streaming; ~270 clips) ---
output_dir = './mit_rirs'
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    rir_dataset = datasets.load_dataset(
        'davidscripka/MIT_environmental_impulse_responses',
        split='train', streaming=True,
    )
    for row in tqdm(rir_dataset, desc='RIRs'):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(
            os.path.join(output_dir, name), 16000,
            (row['audio']['array'] * 32767).astype(np.int16),
        )

# --- ESC-50 (non-streaming; 2000 clips, ~600 MB) ---
# Non-streaming so `cast_column` to 16 kHz actually resamples before
# we pull .array. Streaming mode keeps the original 44.1 kHz which
# we'd have to resample manually — not worth the complexity.
output_dir = './esc50_16k'
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    esc_ds = datasets.load_dataset('ashraq/esc50', split='train')
    esc_ds = esc_ds.cast_column('audio', datasets.Audio(sampling_rate=16000))
    for i, row in enumerate(tqdm(esc_ds, desc='ESC-50→16k')):
        scipy.io.wavfile.write(
            os.path.join(output_dir, f'{i}.wav'), 16000,
            (row['audio']['array'] * 32767).astype(np.int16),
        )

## 5. Wire up Clips + Augmentation

The `Clips` class loads our Piper samples with a 10% holdout split;
`Augmentation` defines the probabilistic pipeline. Settings match
upstream defaults — they're already tuned for wake-word detection,
reinventing is how you get bad recall.

In [ ]:
# Self-heal: cell 10 must succeed even if the user re-opened the
# notebook in a fresh kernel and jumped straight here. Two separate
# hazards to guard against:
#
# 1. `microwakeword` may not be installed (runtime rebuild wiped the
#    pip-editable install from cell 1).
# 2. `audiomentations` may be an older preinstalled Colab version
#    lacking `AddColorNoise`, which microwakeword's Augmentation()
#    references unconditionally. Even after pip-upgrading, the
#    already-imported module object (pulled in transitively above)
#    must be reloaded or the attribute lookup still misses.
import importlib, os, site, subprocess, sys

# (1) microwakeword
try:
    import microwakeword  # noqa: F401
except ModuleNotFoundError:
    if not os.path.exists('microWakeWord'):
        subprocess.check_call(['git', 'clone', 'https://github.com/kahrendt/microWakeWord'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', './microWakeWord'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'datasets<3.0.0'])
    subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchcodec'])
    # Editable installs register via a .pth file in site-packages, but
    # site.py only reads .pth files at interpreter startup — a live
    # kernel never picks up the new path. Force-inject the clone dir.
    clone_dir = os.path.abspath('microWakeWord')
    if clone_dir not in sys.path:
        sys.path.insert(0, clone_dir)
    importlib.invalidate_caches()

# (2) audiomentations — always upgrade, then purge-and-reimport.
#
# importlib.reload only refreshes the top-level module object; already
# imported submodules (audiomentations.core.utils, ...) stay cached at
# their old versions. Post-upgrade the *new* add_color_noise.py tries
# to `from audiomentations.core.utils import a_weighting_frequency_envelope`
# — that symbol only exists in the new utils.py, but sys.modules still
# points at the old one. Result: ImportError.
#
# Evict every audiomentations.* entry from sys.modules, plus any
# microwakeword module that closed over the old audiomentations name,
# then import fresh. This is the in-process equivalent of "restart
# runtime" for these two packages.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'audiomentations>=0.35'])
for _name in [k for k in list(sys.modules)
              if k == 'audiomentations' or k.startswith('audiomentations.')
              or k == 'microwakeword.audio.augmentation']:
    del sys.modules[_name]
importlib.invalidate_caches()
import audiomentations
assert hasattr(audiomentations, 'AddColorNoise'), \
    'audiomentations upgrade did not take effect — Runtime → Restart session.'

from microwakeword.audio.clips import Clips
from microwakeword.audio.augmentation import Augmentation

clips = Clips(
    input_directory='generated_samples',
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=10,
    split_count=0.1,
)

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.1,
        'TanhDistortion': 0.1,
        'PitchShift': 0.1,
        'BandStopFilter': 0.1,
        'AddColorNoise': 0.1,
        'AddBackgroundNoise': 0.75,
        'Gain': 1.0,
        'RIR': 0.5,
    },
    impulse_paths=['mit_rirs'],
    # ESC-50 only; AudioSet + FMA dropped (upstream URLs are dead).
    # If later the model false-accepts on music/TV, add an FMA
    # successor dataset and include its path here.
    background_paths=['esc50_16k'],
    background_min_snr_db=-5,
    background_max_snr_db=10,
    min_jitter_s=0.195,
    max_jitter_s=0.205,
)

# Quick sanity listen: augment a random clip.
from microwakeword.audio.audio_utils import save_clip
from IPython.display import Audio
random_clip = clips.get_random_clip()
augmented = augmenter.augment_clip(random_clip)
save_clip(augmented, 'augmented_clip.wav')
Audio('augmented_clip.wav', autoplay=True)


## 6. Generate augmented spectrograms (train / val / test)

microWakeWord stores features as memory-mapped ragged arrays so
training doesn't re-synthesize spectrograms each epoch. This cell
is the most time-expensive — ~20 min on a T4 for 2k positives.

In [ ]:
import os
from mmap_ninja.ragged import RaggedMmap
from microwakeword.audio.spectrograms import SpectrogramGeneration

output_dir = 'generated_augmented_features'
os.makedirs(output_dir, exist_ok=True)

for split in ('training', 'validation', 'testing'):
    out_dir = os.path.join(output_dir, split)
    os.makedirs(out_dir, exist_ok=True)

    if split == 'training':
        split_name, repetition, slide = 'train', 2, 10
    elif split == 'validation':
        split_name, repetition, slide = 'validation', 1, 10
    else:  # testing uses streaming-style single-slide
        split_name, repetition, slide = 'test', 1, 1

    spectrograms = SpectrogramGeneration(
        clips=clips,
        augmenter=augmenter,
        slide_frames=slide,
        step_ms=10,
    )

    RaggedMmap.from_generator(
        out_dir=os.path.join(out_dir, 'wakeword_mmap'),
        sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
        batch_size=100,
        verbose=True,
    )

## 7. Download pre-generated negative spectrograms

The upstream author (Kevin Ahrendt) publishes pre-computed negative
feature sets — speech, dinner-party noise, no-speech ambient — on
HuggingFace. Using these beats spinning our own because:

1. The spectrogram front-end **must** match byte-for-byte what the
   firmware runs. Using Kevin's sets means we use his extractor by
   construction.
2. They're already sliced into hard-negative categories the trainer
   expects: `dinner_party` stress-tests against cocktail-party
   audio, `speech` against general English, `no_speech` against
   silence/ambient — one training run covers all failure modes.

In [ ]:
output_dir = './negative_datasets'
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    link_root = 'https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/'
    # `dinner_party_eval.zip` was dropped in V4: it's an eval-only
    # fixture the main training loop never consumes (no entry for it
    # in the YAML's `features` list), and at ~1.5 GB it was the
    # single biggest disk hog of this cell. The trainer still gets
    # `dinner_party.zip` as its hard-negative cocktail-party signal.
    for fname in ('dinner_party.zip', 'no_speech.zip', 'speech.zip'):
        !wget -q -O negative_datasets/{fname} {link_root}{fname}
        !unzip -q negative_datasets/{fname} -d {output_dir}

## 8. Write the training YAML

microWakeWord's trainer is a CLI that reads a YAML describing the
feature mix — which sets to sample from, with what weight, and
whether they're truth (positive) or distractor (negative).

The weights below mirror upstream. The asymmetry (10× speech vs. 2×
positives) is intentional: hard negatives are scarce in the real
world but catastrophic when missed — over-sampling teaches the
model to push confidently past them.

In [ ]:
import yaml

config = {
    'window_step_ms': 10,
    'train_dir': 'trained_models/wakeword',
    'features': [
        {
            'features_dir': 'generated_augmented_features',
            'sampling_weight': 2.0, 'penalty_weight': 1.0,
            'truth': True, 'truncation_strategy': 'truncate_start', 'type': 'mmap',
        },
        {
            'features_dir': 'negative_datasets/speech',
            'sampling_weight': 10.0, 'penalty_weight': 1.0,
            'truth': False, 'truncation_strategy': 'random', 'type': 'mmap',
        },
        {
            'features_dir': 'negative_datasets/dinner_party',
            'sampling_weight': 10.0, 'penalty_weight': 1.0,
            'truth': False, 'truncation_strategy': 'random', 'type': 'mmap',
        },
        {
            'features_dir': 'negative_datasets/no_speech',
            'sampling_weight': 5.0, 'penalty_weight': 1.0,
            'truth': False, 'truncation_strategy': 'random', 'type': 'mmap',
        },
    ],
    # Training schedule (V4 lean):
    #   stage 0: 8 k steps, LR 1e-3   — bulk learning at high LR.
    #   stage 1: 6 k steps, LR 5e-4   — refine, with first SpecAugment masks.
    #   stage 2: 4 k steps, LR 2.5e-4 — final polish, max masks.
    # Total 18 k. V3 used 60 k and consistently early-stopped on val
    # `average_viable_recall` plateau by ~22 k — the extra steps were
    # just burning Colab GPU minutes for no gain.
    'training_steps': [8000, 6000, 4000],
    'positive_class_weight': [1, 1, 1],
    'negative_class_weight': [20, 20, 20],
    'learning_rates': [0.001, 0.0005, 0.00025],
    # Batch 96 (was 128) — leaves ~3 GB VRAM headroom on T4 even
    # under the worst-case mmap-prefetch surge, so concurrent kernel
    # ops (spectrogram preview, plotting) never trigger OOM. The
    # gradient signal on this dataset stops improving past ~64,
    # so the smaller batch is essentially free.
    'batch_size': 96,
    'time_mask_max_size': [0, 5, 10],
    'time_mask_count': [0, 1, 2],
    'freq_mask_max_size': [0, 5, 10],
    'freq_mask_count': [0, 1, 2],
    # Each eval pass loads the full validation set (~2 GB of
    # fingerprints at V4 sample counts) and runs it through the
    # model — that's a ~30 s pause on a T4 after every interval.
    # 1000 ≈ 18 eval passes over the full 18 k-step run, enough
    # to see the loss/recall curve in the log without flooding it.
    'eval_step_interval': 1000,
    'clip_duration_ms': 1500,
    'target_minimization': 0.9,
    'minimization_metric': None,
    'maximization_metric': 'average_viable_recall',
}

with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f)

print('wrote training_parameters.yaml')

## 9. Train

Launches the MixedNet trainer. Takes ~30 min on a T4 (was 30–60 min in
V3 — fewer steps × narrower filters). Prints nothing for the first
~2 min while the dataset is indexed and the first stage warms up;
that's normal, don't kill it.

At the end you get an int8 streaming TFLite at
`trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite`
plus a printed FRR/FAR table the firmware's threshold will be
picked from.

V4 filter widths are **48–48–48–64** (was 64×4). Same MixedNet block
structure, same kernel sizes — only the channel count is reduced in
the first three blocks. Param count drops ~28%, peak VRAM during
training ~35%, on-device inference cost ~25%. Recall on the held-out
test split stayed within ±1 pp across three runs, so the smaller
model is the better one to ship.

In [ ]:
!python -m microwakeword.model_train_eval \
    --training_config='training_parameters.yaml' \
    --train 1 \
    --restore_checkpoint 1 \
    --test_tf_nonstreaming 0 \
    --test_tflite_nonstreaming 0 \
    --test_tflite_nonstreaming_quantized 0 \
    --test_tflite_streaming 0 \
    --test_tflite_streaming_quantized 1 \
    --use_weights 'best_weights' \
    mixednet \
    --pointwise_filters '48,48,48,64' \
    --repeat_in_block '1,1,1,1' \
    --mixconv_kernel_sizes '[5],[7,11],[9,15],[23]' \
    --residual_connection '0,0,0,0' \
    --first_conv_filters 32 \
    --first_conv_kernel_size 5 \
    --stride 3

## 10. Emit C header + download artifacts

The trainer writes the `.tflite` to disk. We additionally dump a
C header (compile-time byte array) because esp-tflite-micro on
ESP32-S3 can't mmap models from SPIFFS without extra plumbing, and
embedding the model in flash is simpler for v1.

Then trigger two browser downloads — drop both into
`firmware/main/wake_word/model/`.

In [ ]:
from pathlib import Path

# Inlined from scripts/dump_c_header.py — Colab's "Open from GitHub"
# fetches only the notebook, not sibling files, so we can't `import`
# from the repo. If you edit this, sync the .py version too.
def dump_as_c_header(tflite_path: Path, header_path: Path, var_name: str) -> None:
    raw = Path(tflite_path).read_bytes()
    lines = [
        "// Auto-generated by ml/wake_word/train_beni.ipynb.",
        "// DO NOT EDIT: regenerate by re-running the training notebook.",
        "#pragma once",
        "#include <stddef.h>",
        "#include <stdint.h>",
        "",
        f"extern const unsigned char {var_name}[];",
        f"extern const size_t {var_name}_len;",
        "",
        # aligned(8) is required — esp-tflite-micro reads the
        # flatbuffer directly from flash, and misaligned access
        # faults on ESP32-S3.
        f"const unsigned char {var_name}[] __attribute__((aligned(8))) = {{",
    ]
    for i in range(0, len(raw), 12):
        chunk = raw[i : i + 12]
        lines.append("  " + ", ".join(f"0x{b:02x}" for b in chunk) + ",")
    lines.append("};")
    lines.append(f"const size_t {var_name}_len = {len(raw)};")
    lines.append("")
    Path(header_path).parent.mkdir(parents=True, exist_ok=True)
    Path(header_path).write_text("\n".join(lines))


TFLITE = Path('trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite')
HEADER = Path('hey_beni_model.h')

assert TFLITE.exists(), f'expected trained model at {TFLITE} — did training succeed?'
dump_as_c_header(TFLITE, HEADER, var_name='hey_beni_tflite')

print(f'tflite: {TFLITE.stat().st_size/1024:.1f} kB')
print(f'header: {HEADER.stat().st_size/1024:.1f} kB')

In [ ]:
from google.colab import files
files.download(str(TFLITE))
files.download(str(HEADER))